# Example 1.2.2 / 4: = or default?

Companion notebook for **1.2.2 Computed Parameters** in the AMPL lab Documentation, section "`=` or `default`?". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## `=` or `default`?

A parameter defined with `=` belongs to the model: the data can't give it a value. From
Python, assigning to it raises a `RuntimeError`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;
param frac > 0;
param market {PROD} >= 0;
param mininv {p in PROD} = frac * market[p];

Writing model.mod


### amplpy: data.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    # data.py
    ampl.set["PROD"] = ["bands", "coils"]
    ampl.param["frac"] = 0.1
    ampl.param["market"] = {"bands": 6000, "coils": 4000}
    ampl.param["mininv"] = {"bands": 900}
except Exception as error:
    print(f"{type(error).__name__}: {error}")

RuntimeError: Error executing "let" command:

	mininv has an = assignment in the model.


### Pure AMPL: data.dat

The same in pure AMPL, with the same `model.mod`. The data file is rejected:

In [5]:
%%writefile data.dat
set PROD := bands coils;
param frac := 0.1;
param market := bands 6000  coils 4000;

Writing data.dat


In [6]:
%%writefile data_2.dat
param mininv := bands 900;

Writing data_2.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    ampl.eval(r"""
data data_2.dat;
""")
except Exception:
    pass  # AMPL has printed the error above

[ERROR] 
	data_2.dat
	line 1 offset 6
	mininv was defined in the model.
	context:  param  >>> mininv <<<  := bands 900;
